# Predicting next-day air quality in Ohio: the simple version

This notebook runs the whole project in plain steps. The code is short, and after each result there is a note on how to read it. It gives the same numbers as the poster.

1. **Load** the Excel data
2. **Describe** it: when and where is the air worst?
3. **Prepare** the clues for the prediction (today's AQI, recent days, weather, season, county)
4. **Compare** several prediction models
5. **Use the tuned model** and check what it relies on

**How to run:** choose **Runtime > Run all**, then upload `AQI_Ohio_2022_2026_with_weather.xlsx` when the first cell asks for it. The whole notebook takes about 2 minutes.

**Words you will see**
- **AQI** (Air Quality Index): 0 to 500, higher is worse. 0-50 is Good, 51-100 Moderate, above 100 is unhealthy for sensitive groups.
- **MAE**: how many AQI points the prediction is off on average. **RMSE**: the same idea, but big misses count extra. For both, smaller is better.
- **R²**: how much of the ups and downs the model explains (0 = nothing, 1 = everything). Bigger is better.
- **Baseline**: the lazy guess "tomorrow will be the same as today". A model is only useful if it beats this.

## 1. Get the data

Run the next cell and use the **Choose files** button to upload the Excel file.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = "AQI_Ohio_2022_2026_with_weather.xlsx"
if not os.path.exists(DATA):                 # only ask if the file is not already here
    from google.colab import files
    uploaded = files.upload()                # a "Choose files" button appears: pick the Excel file
    DATA = next(iter(uploaded))              # remember the name of the file you uploaded
print("Using file:", DATA)

## 2. Look at the data

Each row is one county on one day: its AQI, the main pollutant that day, and the weather.

In [ ]:
df = pd.read_excel(DATA)                                         # read the Excel file
df = df.rename(columns={"county Name": "County"})                # a shorter column name
df = df.sort_values(["County", "Date"]).reset_index(drop=True)   # one county after another, oldest day first

print("Rows:", len(df), "| Counties:", df.County.nunique(), "| From", df.Date.min().date(), "to", df.Date.max().date())
print("Missing values:", df.isna().sum().sum(), "| Duplicate county-days:", df.duplicated(["County", "Date"]).sum())
df[["Date", "County", "AQI", "Defining Parameter"]].head()       # a first look

## 3. Describe: when and where is the air worst?

We group the daily AQI by season and by county. For each group we show the average (**mean**), the middle value (**median**), the spread (**std**) and **p90**, the AQI reached or beaten on 1 day in 10, which shows how bad the bad days get.

In [ ]:
# Put every day into a season
month_to_season = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
                   6: "Summer", 7: "Summer", 8: "Summer", 9: "Fall", 10: "Fall", 11: "Fall"}
df["Season"] = df.Date.dt.month.map(month_to_season)

def p90(x):
    return x.quantile(0.9)        # the AQI reached or beaten on 1 day in 10 (the "bad days")

# Average AQI by season
by_season = df.groupby("Season").AQI.agg(["mean", "median", "std", p90])
by_season = by_season.reindex(["Winter", "Spring", "Summer", "Fall"])
print(by_season.round(1))

# Average AQI by county (only counties with at least one year of data)
by_county = df.groupby("County").AQI.agg(["mean", "median", "std", p90, "count"])
by_county = by_county[by_county["count"] >= 365].sort_values("mean")
print("\nLowest counties:")
print(by_county.head(3).round(1))
print("\nHighest counties:")
print(by_county.tail(3).round(1))

In [ ]:
# Are the differences real, or just luck? (Kruskal-Wallis test: a tiny p-value means real)
from scipy.stats import kruskal

season_groups = [g.AQI for _, g in df.groupby("Season")]
county_groups = [g.AQI for name, g in df.groupby("County") if name in by_county.index]
for label, groups in [("seasons", season_groups), ("counties", county_groups)]:
    p = kruskal(*groups).pvalue
    print(f"Differences between {label} are real (p < 0.001)" if p < 0.001 else f"No clear difference between {label} (p = {p:.3f})")

# Two simple charts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
by_season["mean"].plot.bar(ax=ax1, rot=0, title="Average AQI by season")
pd.concat([by_county["mean"].head(3), by_county["mean"].tail(3)]).plot.barh(ax=ax2, title="Lowest and highest counties")
plt.tight_layout()
plt.show()

**How to read this**
- Summer is the worst season: the average AQI is about 52, and 1 day in 10 reaches about 77 or more. The other seasons average about 41 to 44.
- The highest counties are mostly big urban ones (Cuyahoga, Butler, Hamilton). Rural Belmont and Athens are the lowest.
- A tiny p-value (below 0.001) means the differences are real and not just luck.

## 4. Prepare the clues

To predict tomorrow's AQI, the model gets clues that are known **today** for each county. The answer it must learn to predict (the **target**) is tomorrow's AQI in the same county.

| Clue | Plain meaning |
|---|---|
| `aqi_today`, `aqi_yesterday`, `aqi_2_days_ago`, `aqi_6_days_ago` | the county's AQI on those days |
| `avg_last_3_days`, `avg_last_7_days` | the recent average AQI |
| `day_of_year_sin`, `day_of_year_cos` | the time of year (written as a circle, so December is next to January) |
| `day_of_week` | the weekday of the day we are predicting |
| `ozone_is_main` | 1 if ozone is today's main pollutant, otherwise 0 |
| `wind_dir_sin`, `wind_dir_cos` | the wind direction (also written as a circle) |
| `county_average` | the county's usual AQI, learned from the training years only |
| `temp_max`, `temp_mean`, `precip`, `wind_speed_max`, `wind_gust_max`, `humidity`, `solar_radiation` | today's weather |

In [ ]:
county_rows = df.groupby("County")      # shifts and averages must never mix two counties

# The answer we want to predict: TOMORROW's AQI in the same county
df["target"] = county_rows.AQI.shift(-1)
df["next_day_ok"] = (county_rows.Date.shift(-1) - df.Date).dt.days == 1     # is the next row really the next day?

# Clues about the air quality so far
df["aqi_today"] = df.AQI
df["aqi_yesterday"] = county_rows.AQI.shift(1)
df["aqi_2_days_ago"] = county_rows.AQI.shift(2)
df["aqi_6_days_ago"] = county_rows.AQI.shift(6)
df["avg_last_3_days"] = county_rows.AQI.transform(lambda s: s.rolling(3).mean())    # today and the 2 days before
df["avg_last_7_days"] = county_rows.AQI.transform(lambda s: s.rolling(7).mean())    # today and the 6 days before

In [ ]:
# Clues about the calendar. sin and cos turn the year into a circle, so 31 Dec sits next to 1 Jan.
day_of_year = df.Date.dt.dayofyear
df["day_of_year_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
df["day_of_year_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)
df["day_of_week"] = (df.Date + pd.Timedelta(days=1)).dt.dayofweek      # weekday of the day we predict (0 = Monday)

# Clues about the pollutant and the wind (wind direction is a circle too: 359 degrees is next to 0)
df["ozone_is_main"] = (df["Defining Parameter"] == "Ozone").astype(int)    # 1 if ozone is today's main pollutant
wind = np.deg2rad(df["Wind Direction (°)"])
df["wind_dir_sin"] = np.sin(wind)
df["wind_dir_cos"] = np.cos(wind)

# Weather columns: give them shorter names
df = df.rename(columns={"Temp Max (°F)": "temp_max", "Temp Mean (°F)": "temp_mean", "Precipitation (in)": "precip",
                        "Wind Speed Max (mph)": "wind_speed_max", "Wind Gust Max (mph)": "wind_gust_max",
                        "Relative Humidity (%)": "humidity", "Solar Radiation (MJ/m²)": "solar_radiation"})

# Keep only the days where every clue exists and tomorrow is really the next day
ready = df.dropna(subset=["target", "aqi_6_days_ago", "avg_last_7_days"])
ready = ready[ready.next_day_ok]
print("Days we can use:", len(ready))

## 5. Learn from the past, test on the future

We never shuffle the days at random, because that would let the model peek at the future. We **train** on the days before 2025 and **test** on 2025 and later. The test days are used only once, for grading.

In [ ]:
train = ready[ready.Date < "2025-01-01"]      # 2022 - 2024: the models learn from these days
test = ready[ready.Date >= "2025-01-01"]      # 2025 - May 2026: the models are graded on these days
print(f"Train: {len(train):,} rows ({len(train) / len(ready):.0%})  |  Test: {len(test):,} rows ({len(test) / len(ready):.0%})")

features = ["aqi_today", "aqi_yesterday", "aqi_2_days_ago", "aqi_6_days_ago", "avg_last_3_days", "avg_last_7_days",
            "day_of_year_sin", "day_of_year_cos", "day_of_week", "ozone_is_main", "wind_dir_sin", "wind_dir_cos",
            "county_average", "temp_max", "temp_mean", "precip", "wind_speed_max", "wind_gust_max", "humidity", "solar_radiation"]

def with_county_average(rows, source):
    """Add each county's usual AQI, learned from the `source` rows only (never from the rows being graded)."""
    usual_aqi = source.groupby("County").target.mean()
    return rows.assign(county_average=rows.County.map(usual_aqi).fillna(source.target.mean()))

X_train, y_train = with_county_average(train, train)[features], train.target     # clues and answers to learn from
X_test, y_test = with_county_average(test, train)[features], test.target         # clues and answers for grading
print(len(features), "clues per day")

## 6. Compare the models

- **Baseline**: tomorrow = today.
- **Ridge**: a straight-line formula (a simple model).
- **Random Forest** (A and B): many decision trees that vote. This is *bagging*. A uses the default settings, B uses settings chosen by hand.
- **Hist Gradient Boosting**: trees added one after another, each fixing the mistakes of the last. This is *boosting*.

Random Forest and Gradient Boosting are *ensembles*: many small models working together.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def score(predictions):
    """How far off were the predictions on the test days?"""
    return {"MAE": mean_absolute_error(y_test, predictions),
            "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
            "R2": r2_score(y_test, predictions)}

models = {
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Random Forest A": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Random Forest B": RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=10,
                                             max_features=0.5, random_state=42, n_jobs=-1),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=42),
}

results = {"Baseline": score(X_test.aqi_today)}          # the lazy guess: tomorrow = today
for name, model in models.items():
    model.fit(X_train, y_train)                          # learn from the past
    results[name] = score(model.predict(X_test))         # grade on the future
pd.DataFrame(results).T.round(2)

**How to read this**
- Every real model has a lower error than the baseline, so the clues really help.
- The two Random Forests and Gradient Boosting are close to each other. Differences of about 0.1 are small.

## 7. The tuned Random Forest

Tuning means trying many settings and keeping the best one. That search is in the long notebook (`AQI_Ohio_Colab.ipynb`): it tried 25 random combinations of settings, and judged each one on later time periods it had not learned from, **never on the test days**.

Here we simply use the winning settings (400 trees, depth 16, at least 10 days per leaf, 70% of the clues per split).

In [ ]:
best_params = {"n_estimators": 400,        # number of trees
               "max_depth": 16,            # how deep each tree may grow
               "min_samples_leaf": 10,     # smallest group of days allowed at the end of a branch
               "max_features": 0.7}        # share of the clues each split may look at

# The search in the long notebook put the days oldest-first and the county average last.
# We do the same here, so that the numbers match the poster exactly.
oldest_first = train.sort_values(["Date", "County"])
clue_order = [f for f in features if f != "county_average"] + ["county_average"]
X_train_tuned = with_county_average(oldest_first, train)[clue_order]
X_test_tuned = with_county_average(test, train)[clue_order]

tuned = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1).fit(X_train_tuned, oldest_first.target)
results["Random Forest, tuned"] = score(tuned.predict(X_test_tuned))

## 8. Final comparison

In [ ]:
table = pd.DataFrame(results).T
print(table.round(2))

best_name = table.RMSE.idxmin()
improvement = 1 - table.RMSE[best_name] / table.RMSE["Baseline"]
print(f"\nLowest error: {best_name}. Its RMSE is {improvement:.0%} lower than the baseline.")

fig, ax = plt.subplots(figsize=(8, 3.5))
table.RMSE.plot.barh(ax=ax, title="Prediction error (RMSE, smaller is better)")
ax.invert_yaxis()                      # list the models from top to bottom, as in the table
plt.show()

**How to read this**
- The best models cut the error by roughly 17% compared with the lazy baseline.
- The tuned forest is only about as good as Random Forest B. On this data, tuning adds almost nothing: the clues matter more than the settings.

## 9. What does the model rely on, and where does it fail?

We shuffle one clue at a time. The more the error grows, the more the model relied on that clue.

In [ ]:
from sklearn.inspection import permutation_importance

shuffled = permutation_importance(tuned, X_test_tuned, y_test, n_repeats=5, random_state=0,
                                  scoring="neg_root_mean_squared_error", n_jobs=-1)
importance = pd.Series(shuffled.importances_mean, index=clue_order).sort_values(ascending=False)
print(importance.head(6).round(2))

fig, ax = plt.subplots(figsize=(7, 3.5))
importance.head(6).plot.barh(ax=ax, title="How much the error grows when a clue is shuffled")
ax.invert_yaxis()                      # strongest clue at the top
plt.show()

In [ ]:
predicted = tuned.predict(X_test_tuned)

# The really bad days (AQI over 100): did the model see them coming?
bad_days = (y_test > 100).values
print(f"Test days with AQI over 100: {bad_days.sum()}")
print(f"Of those, days where the model predicted over 100: {(predicted[bad_days] > 100).sum()}")

# One county, summer 2025: actual against predicted
county = by_county.index[-1]                     # the county with the highest average AQI
summer = ((test.County == county) & (test.Date >= "2025-05-01") & (test.Date < "2025-09-01")).values
day_predicted = test.Date[summer] + pd.Timedelta(days=1)       # the model predicts the NEXT day
plt.figure(figsize=(10, 3.5))
plt.plot(day_predicted, y_test[summer], label="Actual")
plt.plot(day_predicted, predicted[summer], label="Predicted")
plt.title(f"{county} County, summer 2025: next-day AQI")
plt.legend()
plt.show()

**How to read this**
- Today's AQI is by far the strongest clue. After that come wind gusts, the county, the time of year and sunshine.
- The model follows normal ups and downs well but underestimates the spikes. It did not flag a single one of the days above 100, so it is **useful on typical days, not as a warning system for spikes**.

## 10. Summary in plain English

- **Where and when:** air quality is worst in summer and in the big urban counties. These differences are real.
- **Prediction:** the best models are off by about 8.6 AQI points on average (MAE), against about 10.3 for the lazy baseline. In RMSE terms that is about 12.0 against 14.6.
- **Best model:** Random Forest B and the tuned Random Forest are tied for best. Tuning did not help much.
- **Limit:** it works on typical days but misses the spikes, so it is not a warning system yet.
- **Next step:** add next-day weather forecasts and wildfire-smoke data to help with the spikes.